In [ ]:
import math
import matplotlib.pyplot as plt
import networkx as nx

# Define warehouse locations and 2D spatial coordinates
warehouse_locations = {
    "Receiving_Area": (0, 0),
    "Storage_A": (2, 3),
    "Storage_B": (4, 1),
    "Sorting_Bay": (5, 4),
    "Packing_Station": (8, 5),
}

# Define weighted graph (Location A, Location B, Cost/Distance)
warehouse_edges = [
    ("Receiving_Area", "Storage_A", 3.61),
    ("Receiving_Area", "Storage_B", 4.12),
    ("Storage_A", "Sorting_Bay", 3.16),
    ("Storage_B", "Sorting_Bay", 3.16),
    ("Storage_B", "Packing_Station", 5.66),
    ("Sorting_Bay", "Packing_Station", 3.16),
]


# Euclidean Distance Heuristic Function
def heuristic(current, goal, coords=warehouse_locations):
    x1, y1 = coords[current]
    x2, y2 = coords[goal]
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


# Build NetworkX Graph
G_warehouse = nx.Graph()
for node, pos in warehouse_locations.items():
    G_warehouse.add_node(node, pos=pos)
for u, v, w in warehouse_edges:
    G_warehouse.add_edge(u, v, weight=w)

# Visualization Script
pos = nx.get_node_attributes(G_warehouse, "pos")
labels = nx.get_edge_attributes(G_warehouse, "weight")
plt.figure(figsize=(8, 5))
nx.draw_networkx_nodes(G_warehouse, pos, node_size=700, node_color="skyblue")
nx.draw_networkx_edges(G_warehouse, pos, width=2)
nx.draw_networkx_labels(G_warehouse, pos, font_size=10, font_weight="bold")
nx.draw_networkx_edge_labels(G_warehouse, pos, edge_labels=labels)
plt.title("Warehouse Robot Navigation Graph")
plt.axis("off")
plt.show()

In [ ]:
# Verification script for heuristic consistency across all directed graph edges
goal = "Packing_Station"
print("Edge Consistency Check:\n" + "-" * 50)
for u, v, cost in warehouse_edges:
    hu = heuristic(u, goal)
    hv = heuristic(v, goal)

    # Check direction 1: u -> v
    lhs_1 = round(hu, 2)
    rhs_1 = round(cost + hv, 2)
    valid_1 = lhs_1 <= rhs_1

    # Check direction 2: v -> u
    lhs_2 = round(hv, 2)
    rhs_2 = round(cost + hu, 2)
    valid_2 = lhs_2 <= rhs_2

    print(f"Edge ({u} -> {v}): h({u})={lhs_1} <= {cost} + h({v})={rhs_1} -> {valid_1}")
    print(f"Edge ({v} -> {u}): h({v})={lhs_2} <= {cost} + h({u})={rhs_2} -> {valid_2}")

Task 2

In [ ]:
import heapq
import math
import matplotlib.pyplot as plt
import networkx as nx

# Airport Layout Coordinates
airport_locations = {
    "Baggage_Area": (0, 0),
    "Checkin_Counter": (1, 4),
    "Security_Check": (2, 1),
    "DutyFree_Shop": (5, 5),
    "Transit_Lounge": (4, 2),
    "Boarding_Zone": (7, 4),
    "Departure_Gate": (8, 6),
}

# Graph Edges (u, v, cost)
airport_edges = [
    ("Baggage_Area", "Checkin_Counter", 4.12),
    ("Baggage_Area", "Security_Check", 2.24),
    ("Checkin_Counter", "DutyFree_Shop", 5.00),
    ("Security_Check", "Transit_Lounge", 2.24),
    ("DutyFree_Shop", "Departure_Gate", 3.16),
    ("Transit_Lounge", "Boarding_Zone", 3.16),
    ("Boarding_Zone", "Departure_Gate", 2.24),
]


def heuristic_airport(node, goal="Departure_Gate"):
    x1, y1 = airport_locations[node]
    x2, y2 = airport_locations[goal]
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


def gbfs(graph, start, goal):
    # Priority Queue stores: (h(node), current_node)
    pq = [(heuristic_airport(start, goal), start)]
    came_from = {start: None}
    expansion_sequence = []
    visited = set()

    while pq:
        _, current = heapq.heappop(pq)

        if current in visited:
            continue

        visited.add(current)
        expansion_sequence.append(current)

        if current == goal:
            break

        for neighbor in graph[current]:
            if neighbor not in visited:
                if neighbor not in came_from:
                    came_from[neighbor] = current
                heapq.heappush(
                    pq, (heuristic_airport(neighbor, goal), neighbor)
                )

    # Reconstruct path
    path = []
    curr = goal
    while curr is not None:
        path.append(curr)
        curr = came_from.get(curr)
    path.reverse()

    # Calculate actual path cost
    total_cost = 0
    for i in range(len(path) - 1):
        total_cost += graph[path[i]][path[i + 1]]["weight"]

    return expansion_sequence, path, round(total_cost, 2)


# Construct NetworkX Graph
G_airport = nx.Graph()
for node, pos in airport_locations.items():
    G_airport.add_node(node, pos=pos)
for u, v, w in airport_edges:
    G_airport.add_edge(u, v, weight=w)

# Run GBFS
exp_seq, path_found, cost = gbfs(G_airport, "Baggage_Area", "Departure_Gate")

print(f"Node Expansion Sequence: {exp_seq}")
print(f"Solution Path: {' -> '.join(path_found)}")
print(f"Total Path Cost: {cost}")

Visulaization

In [ ]:
pos = nx.get_node_attributes(G_airport, "pos")
path_edges = list(zip(path_found[:-1], path_found[1:]))

plt.figure(figsize=(9, 6))
nx.draw_networkx_nodes(G_airport, pos, node_size=600, node_color="lightgray")
nx.draw_networkx_nodes(
    G_airport, pos, nodelist=path_found, node_size=700, node_color="orange"
)
nx.draw_networkx_edges(G_airport, pos, width=1.5, edge_color="gray")
nx.draw_networkx_edges(
    G_airport, pos, edgelist=path_edges, width=3, edge_color="red"
)
nx.draw_networkx_labels(G_airport, pos, font_size=9, font_weight="bold")
labels = nx.get_edge_attributes(G_airport, "weight")
nx.draw_networkx_edge_labels(G_airport, pos, edge_labels=labels)
plt.title("GBFS Airport Baggage Handling Path (Red)")
plt.axis("off")
plt.show()

Task 3

In [ ]:
import heapq
import math
import matplotlib.pyplot as plt
import networkx as nx

# Hospital Coordinates
hospital_locations = {
    "Pharmacy": (0, 0),
    "Corridor_A": (1, 4),
    "Corridor_B": (2, 1),
    "ICU": (5, 5),
    "Triage": (4, 2),
    "Radiology": (7, 4),
    "Emergency_Ward": (8, 6),
}

# Graph Edges
hospital_edges = [
    ("Pharmacy", "Corridor_A", 4.12),
    ("Pharmacy", "Corridor_B", 2.24),
    ("Corridor_A", "ICU", 5.00),
    ("Corridor_B", "Triage", 2.24),
    ("ICU", "Emergency_Ward", 3.16),
    ("Triage", "Radiology", 3.16),
    ("Radiology", "Emergency_Ward", 2.24),
]


def heuristic_hospital(node, goal="Emergency_Ward"):
    x1, y1 = hospital_locations[node]
    x2, y2 = hospital_locations[goal]
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


def a_star(graph, start, goal):
    # Priority Queue stores: (f(n), current_node)
    pq = [(heuristic_hospital(start, goal), start)]

    g_score = {node: float("inf") for node in graph.nodes()}
    g_score[start] = 0

    f_score = {node: float("inf") for node in graph.nodes()}
    f_score[start] = heuristic_hospital(start, goal)

    came_from = {start: None}
    expansion_sequence = []
    selected_node_table = []

    visited = set()

    while pq:
        _, current = heapq.heappop(pq)

        if current in visited:
            continue

        visited.add(current)
        expansion_sequence.append(current)

        # Record metrics for expanded node
        h_val = heuristic_hospital(current, goal)
        g_val = g_score[current]
        f_val = f_score[current]
        selected_node_table.append(
            {
                "Node": current,
                "g(n)": round(g_val, 2),
                "h(n)": round(h_val, 2),
                "f(n)": round(f_val, 2),
            }
        )

        if current == goal:
            break

        for neighbor in graph.neighbors(current):
            edge_weight = graph[current][neighbor]["weight"]
            tentative_g = g_score[current] + edge_weight

            if tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                h_neigh = heuristic_hospital(neighbor, goal)
                f_score[neighbor] = tentative_g + h_neigh
                heapq.heappush(pq, (f_score[neighbor], neighbor))

    # Reconstruct path
    path = []
    curr = goal
    while curr is not None:
        path.append(curr)
        curr = came_from.get(curr)
    path.reverse()

    return expansion_sequence, path, round(g_score[goal], 2), selected_node_table


# Construct NetworkX Graph
G_hospital = nx.Graph()
for node, pos in hospital_locations.items():
    G_hospital.add_node(node, pos=pos)
for u, v, w in hospital_edges:
    G_hospital.add_edge(u, v, weight=w)

# Run A* Search
exp_seq, path_found, total_cost, node_metrics = a_star(
    G_hospital, "Pharmacy", "Emergency_Ward"
)

Visulization

In [ ]:
pos = nx.get_node_attributes(G_hospital, "pos")
path_edges = list(zip(path_found[:-1], path_found[1:]))

plt.figure(figsize=(9, 6))
nx.draw_networkx_nodes(G_hospital, pos, node_size=600, node_color="lightgreen")
nx.draw_networkx_nodes(
    G_hospital, pos, nodelist=path_found, node_size=700, node_color="green"
)
nx.draw_networkx_edges(G_hospital, pos, width=1.5, edge_color="gray")
nx.draw_networkx_edges(
    G_hospital, pos, edgelist=path_edges, width=3, edge_color="blue"
)
nx.draw_networkx_labels(G_hospital, pos, font_size=9, font_weight="bold")
labels = nx.get_edge_attributes(G_hospital, "weight")
nx.draw_networkx_edge_labels(G_hospital, pos, edge_labels=labels)
plt.title("A* Emergency Supply Robot Path (Blue)")
plt.axis("off")
plt.show()

Task 4

import heapq
import math

# Drone Delivery Network Locations & Coordinates
drone_locations = {
    "Distribution_Center": (0, 0),
    "Waypoint_A": (2, 8),
    "Waypoint_B": (3, 2),
    "Hub_North": (6, 9),
    "Hub_South": (7, 3),
    "Customer_Building": (10, 10),
}

drone_edges = [
    ("Distribution_Center", "Waypoint_A", 8.25),
    ("Distribution_Center", "Waypoint_B", 3.61),
    ("Waypoint_A", "Hub_North", 4.12),
    ("Waypoint_B", "Hub_South", 4.12),
    ("Hub_North", "Customer_Building", 4.12),
    ("Hub_South", "Customer_Building", 7.62),
    ("Waypoint_A", "Customer_Building", 8.25),
]


def drone_heuristic(node, goal="Customer_Building"):
    x1, y1 = drone_locations[node]
    x2, y2 = drone_locations[goal]
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


def weighted_a_star(start, goal, weight=1.0):
    # Construct adjacency
    adj = {n: [] for n in drone_locations}
    for u, v, w in drone_edges:
        adj[u].append((v, w))
        adj[v].append((u, w))

    pq = [(weight * drone_heuristic(start, goal), start)]
    g_score = {node: float("inf") for node in drone_locations}
    g_score[start] = 0
    came_from = {start: None}

    nodes_expanded = 0
    visited = set()

    while pq:
        _, current = heapq.heappop(pq)

        if current in visited:
            continue

        visited.add(current)
        nodes_expanded += 1

        if current == goal:
            break

        for neighbor, edge_cost in adj[current]:
            tentative_g = g_score[current] + edge_cost
            if tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score = tentative_g + weight * drone_heuristic(neighbor, goal)
                heapq.heappush(pq, (f_score, neighbor))

    # Path reconstruction
    path = []
    curr = goal
    while curr is not None:
        path.append(curr)
        curr = came_from.get(curr)
    path.reverse()

    return path, round(g_score[goal], 2), nodes_expanded


# Run for weights w = 1.0, 1.5, 2.0, 3.0
weights = [1.0, 1.5, 2.0, 3.0]
results = []
for w in weights:
    p, c, n = weighted_a_star("Distribution_Center", "Customer_Building", w)
    results.append((w, " -> ".join(p), c, n))